# 03 U-Net Baseline Training (Kaggle)

Kaggle-ready notebook for EchoNet-Dynamic left ventricle frame-level segmentation.


In [ ]:
from pathlib import Path
import importlib.util
import os
import random

required_modules = [
    'torch',
    'cv2',
    'numpy',
    'pandas',
    'matplotlib',
    'tqdm',
]

missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]

if missing_modules:
    raise ModuleNotFoundError(
        'Missing required package(s): ' + ', '.join(missing_modules) + '\n'
        + 'Install them in the active Kaggle notebook environment before continuing.'
    )

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['image.cmap'] = 'gray'

SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
print('All required packages are available.')


In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
DEFAULT_KAGGLE_DATASET_ROOT = Path('/kaggle/input/datasets/ahmetemirarslan/stanford-data/EchoNet-Dynamic')

def resolve_dataset_root() -> Path:
    candidates = [
        DEFAULT_KAGGLE_DATASET_ROOT,
        Path('/kaggle/input/stanford-data/EchoNet-Dynamic'),
        Path('/kaggle/input/echonet-dynamic/EchoNet-Dynamic'),
        NOTEBOOK_DIR / 'segmentation' / 'data' / 'EchoNet-Dynamic',
        NOTEBOOK_DIR.parent / 'segmentation' / 'data' / 'EchoNet-Dynamic',
    ]

    for root in candidates:
        if (root / 'FileList.csv').exists() and (root / 'VolumeTracings.csv').exists() and (root / 'Videos').exists():
            return root

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for match in sorted(kaggle_input.rglob('FileList.csv')):
            root = match.parent
            if (root / 'VolumeTracings.csv').exists() and (root / 'Videos').exists():
                return root

    raise FileNotFoundError('Could not resolve EchoNet-Dynamic dataset root.')

dataset_root = resolve_dataset_root()
filelist_path = dataset_root / 'FileList.csv'
tracings_path = dataset_root / 'VolumeTracings.csv'
videos_dir = dataset_root / 'Videos'

if str(dataset_root).startswith('/kaggle/input'):
    SEGMENTATION_ROOT = Path('/kaggle/working/segmentation')
else:
    SEGMENTATION_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR / 'segmentation'

RESULTS_ROOT = SEGMENTATION_ROOT / 'results'
FIGURES_DIR = RESULTS_ROOT / 'figures'
METRICS_DIR = RESULTS_ROOT / 'metrics'
MODELS_DIR = RESULTS_ROOT / 'models'

for path in [FIGURES_DIR, METRICS_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RUN_NAME = 'unet_baseline_112_rgb_kaggle'

FRAME_HEIGHT = 112
FRAME_WIDTH = 112
IMAGE_SHAPE = (FRAME_HEIGHT, FRAME_WIDTH)

def resolve_training_device():
    if not torch.cuda.is_available():
        return torch.device('cpu'), 'CUDA is not available in this runtime.'

    try:
        device = torch.device('cuda')
        probe_model = nn.Sequential(
            nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(1),
            nn.ReLU(inplace=True),
        ).to(device)
        probe_input = torch.zeros(1, 1, 8, 8, device=device)
        with torch.no_grad():
            _ = probe_model(probe_input)

        gpu_name = torch.cuda.get_device_name(0)
        del probe_model, probe_input
        torch.cuda.empty_cache()
        return device, f'Using CUDA on {gpu_name}'
    except Exception as exc:
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

        reason = f'{type(exc).__name__}: {exc}'
        return torch.device('cpu'), 'CUDA detected but unusable in this runtime. Falling back to CPU. Reason: ' + reason

DEVICE, DEVICE_MESSAGE = resolve_training_device()
USE_CUDA = DEVICE.type == 'cuda'

BATCH_SIZE = 32 if USE_CUDA else 8
NUM_WORKERS = 2 if USE_CUDA else 0
PIN_MEMORY = USE_CUDA
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
BASE_CHANNELS = 32

REBUILD_FRAME_LEVEL_DF = False

FRAME_LEVEL_RAW_PATH = METRICS_DIR / 'frame_level_df_raw.csv'
FRAME_LEVEL_MERGED_PATH = METRICS_DIR / 'frame_level_df_with_metadata.csv'
CHECKPOINT_PATH = MODELS_DIR / f'{RUN_NAME}_best.pt'
HISTORY_PATH = METRICS_DIR / f'{RUN_NAME}_history.csv'
CURVE_FIG_PATH = FIGURES_DIR / f'{RUN_NAME}_training_curves.png'
PRED_FIG_PATH = FIGURES_DIR / f'{RUN_NAME}_val_predictions.png'

print('dataset_root:', dataset_root)
print('videos_dir:', videos_dir)
print('results_root:', RESULTS_ROOT)
print('checkpoint_path:', CHECKPOINT_PATH)
print('device_resolution:', DEVICE_MESSAGE)


In [ ]:
assert filelist_path.exists(), f'Missing file: {filelist_path}'
assert tracings_path.exists(), f'Missing file: {tracings_path}'
assert videos_dir.exists(), f'Missing directory: {videos_dir}'

file_df = pd.read_csv(filelist_path)
trace_df = pd.read_csv(tracings_path)
trace_df['FileStem'] = trace_df['FileName'].str.replace('.avi', '', regex=False)

file_names = set(file_df['FileName'].astype(str))
trace_names = set(trace_df['FileStem'].astype(str))

traceable_video_names = sorted(file_names & trace_names)
missing_trace_video_names = sorted(file_names - trace_names)
extra_trace_video_names = sorted(trace_names - file_names)

metadata_size_counts = (
    file_df.groupby(['FrameHeight', 'FrameWidth'])
    .size()
    .sort_values(ascending=False)
)
weird_metadata_df = file_df[
    (file_df['FrameHeight'] != FRAME_HEIGHT) |
    (file_df['FrameWidth'] != FRAME_WIDTH)
].copy()
weird_metadata_df['HasTrace'] = weird_metadata_df['FileName'].isin(trace_names)

assert len(traceable_video_names) == 10024
assert len(missing_trace_video_names) == 6
assert len(extra_trace_video_names) == 1
assert weird_metadata_df['HasTrace'].sum() == 0

print('file_df shape:', file_df.shape)
print('trace_df shape:', trace_df.shape)
print('traceable videos:', len(traceable_video_names))
print('missing trace videos:', len(missing_trace_video_names))
print('extra trace-only videos:', len(extra_trace_video_names))
print('metadata size counts:')
display(metadata_size_counts)
print('videos with non-112 metadata (they are outside the traceable segmentation set):')
display(weird_metadata_df[['FileName', 'Split', 'FrameHeight', 'FrameWidth', 'HasTrace']])

display(file_df.head())
display(trace_df.head())


In [ ]:
def resolve_video_path(file_name: str, videos_dir: Path) -> Path:
    candidate_1 = videos_dir / file_name
    candidate_2 = videos_dir / f'{file_name}.avi'

    if candidate_1.exists():
        return candidate_1
    if candidate_2.exists():
        return candidate_2

    raise FileNotFoundError(f'Could not resolve video path for: {file_name}')


def load_frame(video_path: Path, frame_index: int) -> np.ndarray:
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
    ok, frame = cap.read()
    cap.release()

    if not ok:
        raise ValueError(f'Could not read frame {frame_index} from {video_path}')

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return frame


def get_video_trace_rows(file_stem: str, trace_df: pd.DataFrame) -> pd.DataFrame:
    rows = trace_df[trace_df['FileStem'] == file_stem].copy()
    if rows.empty:
        raise ValueError(f'No tracing rows found for {file_stem}')
    return rows


def get_traced_frames_for_video(file_stem: str, trace_df: pd.DataFrame):
    rows = get_video_trace_rows(file_stem, trace_df)
    frames = sorted(rows['Frame'].unique().tolist())
    return frames


def get_frame_trace_points(file_stem: str, frame_index: int, trace_df: pd.DataFrame) -> np.ndarray:
    rows = trace_df[
        (trace_df['FileStem'] == file_stem) &
        (trace_df['Frame'] == frame_index)
    ].copy()

    if rows.empty:
        raise ValueError(f'No trace rows found for {file_stem} frame {frame_index}')

    points = rows[['X1', 'Y1', 'X2', 'Y2']].to_numpy(dtype=np.float32)
    return points


def tracing_points_to_polygon_vertices(points: np.ndarray) -> np.ndarray:
    x1 = points[:, 0]
    y1 = points[:, 1]
    x2 = points[:, 2]
    y2 = points[:, 3]

    x = np.concatenate((x1[1:], np.flip(x2[1:])))
    y = np.concatenate((y1[1:], np.flip(y2[1:])))

    vertices = np.stack([x, y], axis=1)
    return vertices


def polygon_vertices_to_mask(vertices: np.ndarray, image_shape: tuple[int, int]) -> np.ndarray:
    height, width = image_shape

    polygon_points = np.rint(vertices).astype(np.int32).reshape(-1, 1, 2)
    polygon_points[:, 0, 0] = np.clip(polygon_points[:, 0, 0], 0, width - 1)
    polygon_points[:, 0, 1] = np.clip(polygon_points[:, 0, 1], 0, height - 1)

    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(mask, [polygon_points], 1)
    return mask


def build_mask_from_points(points: np.ndarray, image_shape: tuple[int, int]) -> np.ndarray:
    vertices = tracing_points_to_polygon_vertices(points)
    mask = polygon_vertices_to_mask(vertices, image_shape=image_shape)
    return mask


def build_mask_for_video_frame(
    file_stem: str,
    frame_index: int,
    trace_df: pd.DataFrame,
    image_shape: tuple[int, int],
):
    points = get_frame_trace_points(file_stem, frame_index, trace_df)
    vertices = tracing_points_to_polygon_vertices(points)
    mask = polygon_vertices_to_mask(vertices, image_shape=image_shape)
    return points, vertices, mask


In [ ]:
trace_lookup = {
    (str(file_stem), int(frame_idx)): rows[['X1', 'Y1', 'X2', 'Y2']].to_numpy(dtype=np.float32)
    for (file_stem, frame_idx), rows in trace_df.groupby(['FileStem', 'Frame'], sort=False)
}

video_path_map = {
    file_stem: resolve_video_path(file_stem, videos_dir)
    for file_stem in traceable_video_names
}

if FRAME_LEVEL_RAW_PATH.exists() and not REBUILD_FRAME_LEVEL_DF:
    frame_level_df = pd.read_csv(FRAME_LEVEL_RAW_PATH)
    print('Loaded cached raw frame_level_df from:', FRAME_LEVEL_RAW_PATH)
else:
    frame_level_rows = []

    for file_stem in tqdm(traceable_video_names, desc='Building frame_level_df'):
        frames = get_traced_frames_for_video(file_stem, trace_df)
        frame_area_pairs = []

        for frame_idx in frames:
            points = trace_lookup[(file_stem, int(frame_idx))]
            mask = build_mask_from_points(points, IMAGE_SHAPE)
            frame_area_pairs.append((int(frame_idx), int(mask.sum())))

        frame_area_pairs = sorted(frame_area_pairs, key=lambda x: x[1])

        if len(frame_area_pairs) != 2:
            raise ValueError(
                f'Expected exactly 2 traced frames for {file_stem}, got {len(frame_area_pairs)}'
            )

        es_frame, es_area = frame_area_pairs[0]
        ed_frame, ed_area = frame_area_pairs[1]

        frame_level_rows.append({
            'FileName': file_stem,
            'Frame': es_frame,
            'Phase': 'ES',
            'MaskArea': es_area,
        })
        frame_level_rows.append({
            'FileName': file_stem,
            'Frame': ed_frame,
            'Phase': 'ED',
            'MaskArea': ed_area,
        })

    frame_level_df = pd.DataFrame(frame_level_rows)
    frame_level_df.to_csv(FRAME_LEVEL_RAW_PATH, index=False)
    print('Saved raw frame_level_df to:', FRAME_LEVEL_RAW_PATH)

frame_level_df['Frame'] = frame_level_df['Frame'].astype(int)
frame_level_df['MaskArea'] = frame_level_df['MaskArea'].astype(int)

frame_level_df = frame_level_df.merge(
    file_df[['FileName', 'EF', 'ESV', 'EDV', 'FrameHeight', 'FrameWidth', 'FPS', 'NumberOfFrames', 'Split']],
    on='FileName',
    how='left',
    validate='many_to_one',
)

frame_level_df = frame_level_df.sort_values(['Split', 'FileName', 'Frame']).reset_index(drop=True)

expected_split_counts = {'TRAIN': 14920, 'VAL': 2576, 'TEST': 2552}
observed_split_counts = frame_level_df['Split'].value_counts().to_dict()

assert len(frame_level_df) == len(traceable_video_names) * 2
assert observed_split_counts == expected_split_counts, observed_split_counts

frame_level_df.to_csv(FRAME_LEVEL_MERGED_PATH, index=False)

print('trace_lookup size:', len(trace_lookup))
print('video_path_map size:', len(video_path_map))
print('frame_level_df shape:', frame_level_df.shape)
display(frame_level_df.head())
display(frame_level_df['Phase'].value_counts())
display(frame_level_df['Split'].value_counts())
display(pd.crosstab(frame_level_df['Split'], frame_level_df['Phase']))


In [ ]:
sample_rows = frame_level_df.sample(n=min(6, len(frame_level_df)), random_state=SEED)

fig, axes = plt.subplots(2, 3, figsize=(14, 10))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    file_stem = row['FileName']
    frame_idx = int(row['Frame'])

    frame = load_frame(video_path_map[file_stem], frame_idx)
    points = trace_lookup[(file_stem, frame_idx)]
    mask = build_mask_from_points(points, IMAGE_SHAPE)

    ax.imshow(frame)
    ax.imshow(mask, alpha=0.35, cmap='Reds')
    ax.set_title(f"{file_stem}\n{row['Phase']} | frame={frame_idx}")
    ax.axis('off')

for ax in axes[len(sample_rows):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

train_df = frame_level_df[frame_level_df['Split'] == 'TRAIN'].copy().reset_index(drop=True)
val_df = frame_level_df[frame_level_df['Split'] == 'VAL'].copy().reset_index(drop=True)
test_df = frame_level_df[frame_level_df['Split'] == 'TEST'].copy().reset_index(drop=True)

def apply_train_augmentations(image: np.ndarray, mask: np.ndarray):
    image = image.copy()
    mask = mask.copy()

    if np.random.rand() < 0.5:
        image = np.ascontiguousarray(np.fliplr(image))
        mask = np.ascontiguousarray(np.fliplr(mask))

    if np.random.rand() < 0.5:
        alpha = np.random.uniform(0.90, 1.10)
        beta = np.random.uniform(-0.05, 0.05)
        image = np.clip(image * alpha + beta, 0.0, 1.0)

    return image.astype(np.float32), mask.astype(np.float32)

print('train_df shape:', train_df.shape)
print('val_df shape:', val_df.shape)
print('test_df shape:', test_df.shape)
print('TEST split is held out and not used in this notebook.')


In [ ]:
class EchoNetFrameSegmentationDataset(Dataset):
    def __init__(
        self,
        frame_df: pd.DataFrame,
        video_path_map: dict,
        trace_lookup: dict,
        image_shape: tuple[int, int] = (112, 112),
        train: bool = False,
    ):
        self.frame_df = frame_df.reset_index(drop=True).copy()
        self.video_path_map = video_path_map
        self.trace_lookup = trace_lookup
        self.image_shape = image_shape
        self.train = train

    def __len__(self):
        return len(self.frame_df)

    def __getitem__(self, idx: int):
        row = self.frame_df.iloc[idx]

        file_stem = row['FileName']
        frame_idx = int(row['Frame'])

        frame = load_frame(self.video_path_map[file_stem], frame_idx).astype(np.float32) / 255.0
        if frame.shape[:2] != self.image_shape:
            raise ValueError(f'Unexpected frame shape for {file_stem}: {frame.shape[:2]}')

        points = self.trace_lookup[(file_stem, frame_idx)]
        mask = build_mask_from_points(points, image_shape=self.image_shape).astype(np.float32)

        if self.train:
            frame, mask = apply_train_augmentations(frame, mask)

        image_tensor = torch.from_numpy(np.transpose(frame, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask[None, ...]).float()

        return {
            'image': image_tensor,
            'mask': mask_tensor,
            'file_name': file_stem,
            'frame': frame_idx,
            'phase': row['Phase'],
        }

train_dataset = EchoNetFrameSegmentationDataset(
    frame_df=train_df,
    video_path_map=video_path_map,
    trace_lookup=trace_lookup,
    image_shape=IMAGE_SHAPE,
    train=True,
)

val_dataset = EchoNetFrameSegmentationDataset(
    frame_df=val_df,
    video_path_map=video_path_map,
    trace_lookup=trace_lookup,
    image_shape=IMAGE_SHAPE,
    train=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

print('len(train_dataset):', len(train_dataset))
print('len(val_dataset):', len(val_dataset))
print('len(train_loader):', len(train_loader))
print('len(val_loader):', len(val_loader))

batch = next(iter(train_loader))
print('image batch shape:', batch['image'].shape)
print('mask batch shape:', batch['mask'].shape)

n_show = min(3, batch['image'].shape[0])
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))

if n_show == 1:
    axes = [axes]

for i in range(n_show):
    image = batch['image'][i].permute(1, 2, 0).numpy()
    mask = batch['mask'][i, 0].numpy()

    axes[i].imshow(image)
    axes[i].imshow(mask, alpha=0.35, cmap='Reds')
    axes[i].set_title(f"{batch['file_name'][i]}\nframe={int(batch['frame'][i])}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),
            DoubleConv(in_channels, out_channels),
        )

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)

        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)

        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, base_channels: int = 32):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base_channels)
        self.enc2 = DownBlock(base_channels, base_channels * 2)
        self.enc3 = DownBlock(base_channels * 2, base_channels * 4)
        self.enc4 = DownBlock(base_channels * 4, base_channels * 8)
        self.bottleneck = DownBlock(base_channels * 8, base_channels * 16)

        self.dec4 = UpBlock(base_channels * 16, base_channels * 8, base_channels * 8)
        self.dec3 = UpBlock(base_channels * 8, base_channels * 4, base_channels * 4)
        self.dec2 = UpBlock(base_channels * 4, base_channels * 2, base_channels * 2)
        self.dec1 = UpBlock(base_channels * 2, base_channels, base_channels)

        self.out_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)
        x4 = self.enc4(x3)
        x5 = self.bottleneck(x4)

        x = self.dec4(x5, x4)
        x = self.dec3(x, x3)
        x = self.dec2(x, x2)
        x = self.dec1(x, x1)

        return self.out_conv(x)

model = UNet(in_channels=3, out_channels=1, base_channels=BASE_CHANNELS)
dummy_input = torch.zeros(2, 3, FRAME_HEIGHT, FRAME_WIDTH)
dummy_output = model(dummy_input)

print('dummy_input shape:', dummy_input.shape)
print('dummy_output shape:', dummy_output.shape)
print('parameter count:', f"{sum(p.numel() for p in model.parameters()):,}")


In [ ]:
bce_loss_fn = nn.BCEWithLogitsLoss()

def dice_loss_from_logits(logits, targets, smooth: float = 1e-6):
    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)
    intersection = (probs * targets).sum(dim=dims)
    denominator = probs.sum(dim=dims) + targets.sum(dim=dims)

    dice_score = (2.0 * intersection + smooth) / (denominator + smooth)
    return 1.0 - dice_score.mean()


def bce_dice_loss(logits, targets, bce_weight: float = 0.5, dice_weight: float = 0.5):
    bce = bce_loss_fn(logits, targets)
    dice = dice_loss_from_logits(logits, targets)
    total = bce_weight * bce + dice_weight * dice
    return total, bce.detach(), dice.detach()


@torch.no_grad()
def dice_iou_from_logits(logits, targets, threshold: float = 0.5, eps: float = 1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    dims = (1, 2, 3)
    intersection = (preds * targets).sum(dim=dims)
    pred_sum = preds.sum(dim=dims)
    target_sum = targets.sum(dim=dims)
    union = pred_sum + target_sum - intersection

    dice = (2.0 * intersection + eps) / (pred_sum + target_sum + eps)
    iou = (intersection + eps) / (union + eps)

    return dice.mean().item(), iou.mean().item()


In [ ]:
def run_one_epoch(model, loader, device, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    running = {
        'loss': 0.0,
        'bce': 0.0,
        'dice_loss': 0.0,
        'dice': 0.0,
        'iou': 0.0,
    }

    progress_bar = tqdm(loader, desc='train' if is_train else 'val', leave=False)

    for batch in progress_bar:
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss, bce, dice_loss_value = bce_dice_loss(logits, masks)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        batch_dice, batch_iou = dice_iou_from_logits(logits, masks)
        batch_size = images.size(0)

        running['loss'] += loss.item() * batch_size
        running['bce'] += bce.item() * batch_size
        running['dice_loss'] += dice_loss_value.item() * batch_size
        running['dice'] += batch_dice * batch_size
        running['iou'] += batch_iou * batch_size

        progress_bar.set_postfix(
            loss=f'{loss.item():.4f}',
            dice=f'{batch_dice:.4f}',
            iou=f'{batch_iou:.4f}',
        )

    n_samples = len(loader.dataset)
    epoch_metrics = {key: value / n_samples for key, value in running.items()}
    return epoch_metrics


device = DEVICE

model = UNet(
    in_channels=3,
    out_channels=1,
    base_channels=BASE_CHANNELS,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3,
)

best_val_dice = -np.inf
history = []

run_config = {
    'run_name': RUN_NAME,
    'dataset_root': str(dataset_root),
    'frame_height': FRAME_HEIGHT,
    'frame_width': FRAME_WIDTH,
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'base_channels': BASE_CHANNELS,
    'train_examples': len(train_dataset),
    'val_examples': len(val_dataset),
    'device': str(device),
}

print('device:', device)
print('device_message:', DEVICE_MESSAGE)
print('model parameters:', f"{sum(p.numel() for p in model.parameters()):,}")
display(pd.Series(run_config))


In [ ]:
for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = run_one_epoch(
        model=model,
        loader=train_loader,
        device=device,
        optimizer=optimizer,
    )

    val_metrics = run_one_epoch(
        model=model,
        loader=val_loader,
        device=device,
        optimizer=None,
    )

    scheduler.step(val_metrics['dice'])
    current_lr = optimizer.param_groups[0]['lr']

    epoch_row = {
        'epoch': epoch,
        'lr': current_lr,
        'train_loss': train_metrics['loss'],
        'train_bce': train_metrics['bce'],
        'train_dice_loss': train_metrics['dice_loss'],
        'train_dice': train_metrics['dice'],
        'train_iou': train_metrics['iou'],
        'val_loss': val_metrics['loss'],
        'val_bce': val_metrics['bce'],
        'val_dice_loss': val_metrics['dice_loss'],
        'val_dice': val_metrics['dice'],
        'val_iou': val_metrics['iou'],
    }
    history.append(epoch_row)

    print(
        f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"train_dice={train_metrics['dice']:.4f} | "
        f"val_dice={val_metrics['dice']:.4f} | "
        f"val_iou={val_metrics['iou']:.4f} | "
        f'lr={current_lr:.2e}'
    )

    if val_metrics['dice'] > best_val_dice:
        best_val_dice = val_metrics['dice']

        torch.save(
            {
                'epoch': epoch,
                'best_val_dice': best_val_dice,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'run_config': run_config,
                'history': history,
            },
            CHECKPOINT_PATH,
        )

        print(f'Saved new best checkpoint -> {CHECKPOINT_PATH}')


In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)
display(history_df.tail())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train Loss')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='Val Loss')
axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_dice'], label='Train Dice')
axes[1].plot(history_df['epoch'], history_df['val_dice'], label='Val Dice')
axes[1].plot(history_df['epoch'], history_df['train_iou'], label='Train IoU', linestyle='--')
axes[1].plot(history_df['epoch'], history_df['val_iou'], label='Val IoU', linestyle='--')
axes[1].set_title('Segmentation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].legend()

plt.tight_layout()
plt.savefig(CURVE_FIG_PATH, dpi=150, bbox_inches='tight')
plt.show()

best_checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

sample_val_rows = val_df.sample(n=min(6, len(val_df)), random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(14, 10))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample_val_rows.iterrows()):
    file_stem = row['FileName']
    frame_idx = int(row['Frame'])

    frame = load_frame(video_path_map[file_stem], frame_idx).astype(np.float32) / 255.0
    gt_mask = build_mask_from_points(
        trace_lookup[(file_stem, frame_idx)],
        IMAGE_SHAPE,
    ).astype(np.uint8)

    image_tensor = torch.from_numpy(np.transpose(frame, (2, 0, 1))).unsqueeze(0).float().to(device)

    with torch.no_grad():
        pred_logits = model(image_tensor)
        pred_probs = torch.sigmoid(pred_logits)[0, 0].cpu().numpy()

    pred_mask = (pred_probs >= 0.5).astype(np.uint8)

    ax.imshow(frame)
    ax.imshow(np.ma.masked_where(gt_mask == 0, gt_mask), alpha=0.25, cmap='Greens')
    ax.imshow(np.ma.masked_where(pred_mask == 0, pred_mask), alpha=0.25, cmap='Reds')
    ax.set_title(f"{file_stem}\n{row['Phase']} | frame={frame_idx}")
    ax.axis('off')

for ax in axes[len(sample_val_rows):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(PRED_FIG_PATH, dpi=150, bbox_inches='tight')
plt.show()

best_row = history_df.loc[history_df['val_dice'].idxmax()]

artifact_summary = pd.Series({
    'best_epoch': int(best_row['epoch']),
    'best_val_dice': float(best_row['val_dice']),
    'best_val_iou': float(best_row['val_iou']),
    'checkpoint_path': str(CHECKPOINT_PATH),
    'history_path': str(HISTORY_PATH),
    'curve_fig_path': str(CURVE_FIG_PATH),
    'prediction_fig_path': str(PRED_FIG_PATH),
    'frame_level_raw_path': str(FRAME_LEVEL_RAW_PATH),
    'frame_level_merged_path': str(FRAME_LEVEL_MERGED_PATH),
})

display(artifact_summary)
